# 08. Interrupts and Time Travel — Execute interrupt, Acknowledge, Rewind

## Learning Objectives

Execute with `interrupt()`, interrupt, and resume with `Command(resume=...)`. Time travel back to the previous state.

- Human-in-the-loop pattern can be implemented
- Interrupt can also be used in Functional API
- You can perform time travel using checkpoint history.
- state can be modified externally with `update_state()`

## 8.1 Environment Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")
print("Model is ready")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically activated when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 8.2 interrupt() — executes interrupt and waits for human input

- `interrupt(value)`: Save the current state to the checkpoint and execute interrupt
- `Command(resume=value)`: Passes the value at interrupt point and resume

This pattern is used to obtain human approval or input additional information before performing sensitive tasks.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


class ReviewState(TypedDict):
    document: str
    approved: bool
    final_result: str


def draft_document(state: ReviewState) -> dict:
    return {
        "document": f"초안: {state.get('document', '주제')}에 대한 중요 문서"
    }


def human_review(state: ReviewState) -> dict:
    # wait for human approval
    decision = interrupt(
        {
            "document": state["document"],
            "question": "Would you like to approve this document? (yes/no)"
        }
    )

    return {
        "approved": decision == "yes"
    }


def finalize(state: ReviewState) -> dict:
    if state["approved"]:
        return {
            "final_result": f"승인됨: {state['document']}"
        }

    return {
        "final_result": f"거절됨: {state['document']}"
    }


builder = StateGraph(ReviewState)

builder.add_node("draft", draft_document)
builder.add_node("review", human_review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": "review-1"
    }
}

# Step 1: Run → interrupt in review node
result = graph.invoke(
    {
        "document": "AI safety"
    },
    {**config, **lf_config}
)

print("Step 1 - interrupt in review")

state = graph.get_state(config)

print(f"  다음 노드: {state.next}")
print(f"  인터럽트 값: {state.tasks}")

## 8.3 Command(resume=...) — interrupt executes resume

Using `Command(resume=value)` causes execution to resume at the point where `interrupt()` is called. The value passed to `resume` becomes the return value of `interrupt()`.

In [ ]:
# Step 2: Approve resume

result = graph.invoke(
    Command(resume="yes"),
    {**config, **lf_config}
)

print(f"Step 2 - 승인으로 재개됨")
print(f"  결과: {result['final_result']}")

## 8.4 Interrupt in Functional API

You can also use `interrupt()` in the Functional API (`@entrypoint`, `@task`).

In [ ]:
from langgraph.func import entrypoint, task

@task
def generate_proposal(topic: str) -> str:
    response = model.invoke(f"다음에 대한 한 문장 제안을 작성해주세요: {topic}", config=lf_config)
    return response.content

@entrypoint(checkpointer=InMemorySaver())
def proposal_workflow(topic: str) -> dict:
    proposal = generate_proposal(topic).result()

    # waiting for human approval
    approval = interrupt({
        "proposal": proposal,
        "action": "Approve or reject"
    })

    return {
        "proposal": proposal,
        "approved": approval,
    }

config = {"configurable": {"thread_id": "proposal-1"}}

# Run → interrupt
result = proposal_workflow.invoke("renewable energy", {**config, **lf_config})
print("Proposal workflow interrupt received")

# resume
result = proposal_workflow.invoke(Command(resume="approval"), {**config, **lf_config})
print(f"최종: {result}")

## 8.5 Time Travel — Go back to a previous checkpoint

The checkpoint system in LangGraph stores all executions of state. You can view previous checkpoints with `get_state_history()` and go back to a specific point in time.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.messages import HumanMessage

def chatbot(state: MessagesState) -> dict:
    response = model.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "timetravel-1"}}

# 3 conversations
graph.invoke({"messages": [HumanMessage(content="My favorite color is blue.")]}, {**config, **lf_config})
graph.invoke({"messages": [HumanMessage(content="My favorite food is pizza.")]}, {**config, **lf_config})
graph.invoke({"messages": [HumanMessage(content="I live in Seoul.")]}, {**config, **lf_config})

# Select a specific checkpoint in history
history = list(graph.get_state_history(config))
print(f"전체 체크포인트 수: {len(history)}")

# Go back to the second conversation
target = history[2]  # older checkpoint
target_config = target.config
print(f"\n체크포인트에서 재생: {target_config['configurable']['checkpoint_id'][:20]}...")

# At that point start a new conversation
result = graph.invoke(
    {"messages": [HumanMessage(content="What is my favorite food?")]},
    {**target_config, **lf_config},
)
print(f"응답: {result['messages'][-1].content[:200]}")

## 8.6 update_state() — Time travel + state fix

`update_state()` allows you to directly modify the state of a graph from the outside. This is useful for debugging, testing, or when manual intervention is required.

In [ ]:
# Add information to current state
from langchain.messages import AIMessage

graph.update_state(config, {
    "messages": [AIMessage(content="(Update: Users also like cats.)")]
})

result = graph.invoke(
    {"messages": [HumanMessage(content="What do you know about my preferences?")]},
    {**config, **lf_config}
)
print("state After update:", result["messages"][-1].content[:200])

## 8.7 Summary

Summarize the key functions learned in this Note book.

| Features | API | Description |
|------|-----|------|
| `interrupt(value)` | Both sides | run interrupt, pass value |
| `Command(resume=value)` | Both sides | resume at point interrupt |
| `get_state_history()` | Graph | Checkpoint history inquiry |
| `update_state()` | Graph | Modify state externally |

**interrupts and time travel** are key features in production AI applications:
- **interrupt**: Get human approval before sensitive operations
- **Time Travel**: You can go back to the previous state and explore different routes
- **update_state**: You can adjust the execution flow by modifying state externally.

### Next Steps
→ **[09_subgraphs.ipynb](./09_subgraphs.ipynb)**: Learn subgraphs.